# Day 5 · Concept 3 of 5: Plan B and the Human Handoff

**Big idea:** every failure must end somewhere safe. Decide the exit in advance.
*About 15 minutes. No coding needed to follow along.*

---

### The idea in plain words
When the card company cannot help, a good call-centre agent does one of two things:

1. **Plan B (fallback):** "Our system is busy, please call back later." The problem is *temporary*, so we give a safe, honest answer.
2. **Human handoff (escape path):** "I am transferring you to a colleague." The problem is *permanent* (for example you gave a wrong account number), so retrying is pointless.

The skill is telling the two apart. That is called **classifying the error**:

| Kind of error | Example | What to do |
| --- | --- | --- |
| **Temporary (transient)** | line dropped, service busy | Retry, then Plan B |
| **Permanent** | the customer id `123` is not a valid id | Do not retry. Hand off to a human |

### What went wrong in L2
In L2 a failure ended the run, or the agent kept trying the same hopeless thing. The model might even "make up" an answer to fill the silence.

### What we do now (L3)
**Code decides the exit, never the AI model.** Fixed messages, no guessing, and the customer can tell it was a fallback.

In [ ]:
# Step 0 - Connect to the AI model (Ollama). Run this cell first.
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gpt-oss:120b-cloud", temperature=0)
try:
    print("Model says:", llm.invoke("Reply with just the word: ready").content)
except Exception as problem:
    print("The model is not reachable yet.")
    print("Fix: open a terminal, run   ollama signin   then restart this notebook.")
    print("Technical detail:", type(problem).__name__)

### The agent we will build
```
START -> [ call_tool ] --ok-----------------> [ write_reply ] -> END
               |  \--temporary, tries left--> (call again)
               |  \--temporary, out of tries-> [ fallback ]  -> END
               \--permanent------------------> [ escalate ]  -> END
```
We also build a **fire drill**: a table that says what each pretend call does, so every failure is repeatable. This is called *failure injection*.

In [ ]:
import time
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

MAX_TRIES = 2
SCENARIOS = {                       # the fire drills: what the pretend card company does on each call
    "healthy":   ["ok"],
    "flaky":     ["error", "ok"],
    "down":      ["error", "error"],
    "bad_input": ["ok"],            # same healthy service, but we will give it a wrong id
}

class State(TypedDict):
    scenario: str
    user: str
    attempt: int
    error: str
    permanent: bool
    fallback: bool
    balance: str
    reply: str

def call_tool(state):
    attempt = state["attempt"] + 1
    if not (state["user"].startswith("cust-") and state["user"][5:].isdigit()):
        return {"attempt": attempt, "error": "user id must look like cust-123", "permanent": True}   # permanent
    if SCENARIOS[state["scenario"]][attempt - 1] == "error":
        return {"attempt": attempt, "error": "ConnectionError", "permanent": False}                  # temporary
    return {"attempt": attempt, "error": "", "balance": "1,250 INR"}

def router(state):       # classify the error, then choose the exit
    if state["error"] == "":
        return "done"
    if state["permanent"]:
        return "escape"
    if state["attempt"] < MAX_TRIES:
        return "retry"
    return "give_up"

def fallback(state):
    return {"fallback": True, "reply": "Service busy, please try again later."}

def escalate(state):
    return {"reply": "ESCALATED to a human agent. No automatic retry."}

def write_reply(state):
    return {"reply": llm.invoke("You are a polite bank assistant. Say this in one short sentence: the balance is " + state["balance"]).content}

builder = StateGraph(State)
builder.add_node("call_tool", call_tool)
builder.add_node("write_reply", write_reply)
builder.add_node("fallback", fallback)
builder.add_node("escalate", escalate)
builder.add_edge(START, "call_tool")
builder.add_conditional_edges("call_tool", router,
    {"done": "write_reply", "retry": "call_tool", "give_up": "fallback", "escape": "escalate"})
builder.add_edge("write_reply", END)
builder.add_edge("fallback", END)
builder.add_edge("escalate", END)
agent = builder.compile()

def drill(scenario, user="cust-1"):
    result = agent.invoke({"scenario": scenario, "user": user, "attempt": 0, "error": "", "permanent": False,
                           "fallback": False, "balance": "", "reply": ""})
    print(f"{scenario:10} | tries: {result['attempt']} | fallback flag: {result['fallback']!s:5} | customer sees: {result['reply']}")

### Run the four fire drills
Predict the number of tries and the ending for each one **before** you run the cell.

In [ ]:
drill("healthy")
drill("flaky")
drill("down")
drill("bad_input", user="123")      # the id 123 is permanently wrong

**What you should see:**

| Drill | What happened | Exit |
| --- | --- | --- |
| healthy | worked first time | normal answer (model wrote it) |
| flaky | one failure, retry worked | normal answer |
| down | both tries failed | **Plan B**: "Service busy..." and `fallback flag: True` |
| bad_input | wrong id, so only **1** try | **Human handoff**: no retry wasted |

Notice that the fallback and the handoff use **fixed text**. The model is not asked to improvise when something has gone wrong.

### Your turn: predict first, then run
Run the **flaky** drill, but with the wrong id `123`.

**Predict:** does it use 1 try or 2? Which exit does it take?

In [ ]:
drill("flaky", user="123")

**Answer:** 1 try and the human handoff. The id is checked first, and a permanent error never earns a retry.

---

### Remember
* **Temporary** error: retry a little, then Plan B (a safe, honest, visible fallback).
* **Permanent** error: skip the retries, hand off to a human.
* **Code** decides the exit. The model only writes the happy-path sentence.
* Failure injection (the `SCENARIOS` table) lets you rehearse failures like a fire drill.

> **Technical note (optional):** the `fallback` state flag and the sentence "Service busy, please try again later." are exactly what Day 19 `agent.py::write_answer` uses.

**Next notebook:** what if a thousand customers ask at once?